In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import json
from pathlib import Path
import pandas as pd

drive_root = Path("/content/drive")
matches = list(drive_root.glob("**/agropulse_xai"))
BASE_DIR = matches[0] if matches else Path("/content/drive/MyDrive/agropulse_xai")

verif_dir = BASE_DIR / "data" / "parquet_VERIFICADO"
print("=== ARCHIVOS EN parquet_VERIFICADO ===")
for f in sorted(verif_dir.glob("*.parquet")):
    df_tmp = pd.read_parquet(f)
    print(f"- {f.name} | shape={df_tmp.shape} | cols={list(df_tmp.columns)}")

rot_path = BASE_DIR / "data" / "rotations.json"
if rot_path.exists():
    with open(rot_path, "r", encoding="utf-8") as f:
        print("\n=== CONTENIDO DE rotations.json ===")
        print(json.dumps(json.load(f), indent=2))

=== ARCHIVOS EN parquet_VERIFICADO ===
- D2_ssurgo_soil.parquet | shape=(4, 12) | cols=['mukey', 'muname', 'compname', 'comppct_r', 'hzdept_r', 'hzdepb_r', 'materia_organica', 'agua_disponible_awc', 'erodabilidad_k', 'arena', 'limo', 'arcilla']
- aligned_2016_2025.parquet | shape=(3653, 8) | cols=['date', 'year', 'tmax_c', 'tmin_c', 'solar_rad', 'et0_mm', 'smap_rootzone', 'gpm_rain_mm']
- climate_long_1981_2025.parquet | shape=(16436, 7) | cols=['date', 'year', 'tmax_c', 'tmin_c', 'solar_rad', 'precip_power_mm', 'et0_mm']
- harmonized_grid_30m.parquet | shape=(1200, 9) | cols=['pixel_id', 'lon', 'lat', 'elevation_m', 'slope_pct', 'ssurgo_awc', 'smap_sm_30m', 'gpm_precip_30m', 'power_tmax_30m']
- hls_series_2016_2025.parquet | shape=(452666, 9) | cols=['pixel_id', 'date', 'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'fmask']


In [19]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.ndimage import label
from statsmodels.tsa.seasonal import STL

# 1. Localizar rutas exactas en el árbol compartido
drive_root = Path("/content/drive")
EST1_DIR = list(drive_root.glob("**/ESTADISTICO 1"))[0]
VERIF_DIR = list(drive_root.glob("**/parquet_VERIFICADO"))[0]
DATA_DIR = VERIF_DIR.parent
BASE_DIR = DATA_DIR.parent

mat_matches = list(drive_root.glob("**/RESULTADOS A REVISAR (MATEMÁTICO)"))
MAT_DIR = mat_matches[0] if mat_matches else None

print(f"Carpeta raíz: {BASE_DIR}")
print(f"Carpeta revisión Matemático: {MAT_DIR}")
print(f"Carpeta entrega Estadístico 1: {EST1_DIR}\n")

# 2. Detectar la grilla armonizada más reciente
grid_candidates = []
for folder in [MAT_DIR, VERIF_DIR, DATA_DIR]:
    if folder and folder.exists():
        for pq in folder.glob("*.parquet"):
            if "grid" in pq.name.lower() or "harmoniz" in pq.name.lower():
                grid_candidates.append(pq)

grid_path = sorted(grid_candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
df_grid = pd.read_parquet(grid_path)
df_hls_raw = pd.read_parquet(VERIF_DIR / "hls_series_2016_2025.parquet")

print("=== GRILLA ARMONIZADA SELECCIONADA ===")
print(f"Archivo: {grid_path}")
print(f"Píxeles: {len(df_grid)} | Columnas: {list(df_grid.columns)}\n")

rot_candidates = sorted(BASE_DIR.rglob("rotations.json"), key=lambda p: p.stat().st_mtime, reverse=True)
rotations_data = json.loads(rot_candidates[0].read_text(encoding="utf-8")) if rot_candidates else {}

cat_candidates = list(BASE_DIR.rglob("crops_catalog.json"))
catalog = json.loads(cat_candidates[0].read_text(encoding="utf-8")) if cat_candidates else {}

# 3. Enmascaramiento de Nubes, Sombras y NIEVE (Fmask + NDSI) + Alineación 2016-2025
df = df_hls_raw.copy()
df["date"] = pd.to_datetime(df["date"])
df = df[(df["date"].dt.year >= 2016) & (df["date"].dt.year <= 2025)].copy()

eps = 1e-6
blue = df["blue"].to_numpy(dtype=np.float64)
green = df["green"].to_numpy(dtype=np.float64)
red = df["red"].to_numpy(dtype=np.float64)
nir = df["nir"].to_numpy(dtype=np.float64)
swir1 = df["swir1"].to_numpy(dtype=np.float64)
swir2 = df["swir2"].to_numpy(dtype=np.float64)

ndsi = np.clip((green - swir1) / (green + swir1 + eps), -1.0, 1.0)
df["ndsi"] = ndsi

if "fmask" in df.columns:
    fmask_vals = df["fmask"].fillna(255).astype(int).to_numpy()
    fmask_clear = (fmask_vals == 0) | ((fmask_vals & 0b00011110) == 0)
else:
    fmask_clear = np.ones(len(df), dtype=bool)

valid_obs_mask = fmask_clear & (ndsi <= 0.20) & (nir > 0) & (red > 0)
df["is_valid_obs"] = valid_obs_mask.astype(np.float64)

pixel_validity = (
    df.groupby("pixel_id", as_index=False)
    .agg(frac_hls_valida=("is_valid_obs", "mean"))
)
frac_hls_valida_parcel = float(df["is_valid_obs"].mean())

df_clean = df[valid_obs_mask].copy()
c_blue = df_clean["blue"].to_numpy(dtype=np.float64)
c_red = df_clean["red"].to_numpy(dtype=np.float64)
c_nir = df_clean["nir"].to_numpy(dtype=np.float64)
c_swir1 = df_clean["swir1"].to_numpy(dtype=np.float64)
c_swir2 = df_clean["swir2"].to_numpy(dtype=np.float64)

# 4. Índices Espectrales: NDVI, BSI y NDTI (Separación Rastrojo vs Suelo Desnudo)
ndvi = np.clip((c_nir - c_red) / (c_nir + c_red + eps), -1.0, 1.0)
bsi = np.clip(((c_swir1 + c_red) - (c_nir + c_blue)) / ((c_swir1 + c_red) + (c_nir + c_blue) + eps), -1.0, 1.0)
ndti = np.clip((c_swir1 - c_swir2) / (c_swir1 + c_swir2 + eps), -1.0, 1.0)

df_clean["ndvi"] = ndvi
df_clean["bsi"] = bsi
df_clean["ndti"] = ndti
df_clean["month"] = df_clean["date"].dt.month

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

p_bsi = _sigmoid((bsi - 0.05) / 0.04)
p_veg = _sigmoid((0.35 - ndvi) / 0.06)
p_no_residue = _sigmoid((0.12 - ndti) / 0.04)

df_clean["prob_bare_obs"] = np.round(p_bsi * p_veg * p_no_residue, 4)
df_clean["is_residue_cover"] = ((ndti > 0.12) & (ndvi <= 0.35)).astype(np.float64)
df_clean["is_bare_real"] = ((bsi >= 0.05) & (ndvi <= 0.35) & (ndti <= 0.12)).astype(np.float64)

# 5. Cobertura Invernal por Verdor Primaveral Pre-Siembra (Marzo - Abril)
spring_mask = df_clean["month"].isin([3, 4])
spring_vigor = (
    df_clean[spring_mask]
    .groupby("pixel_id", as_index=False)
    .agg(spring_ndvi_p75=("ndvi", lambda s: float(np.percentile(s, 75))))
)
spring_vigor["spring_cover_prob"] = np.round(_sigmoid((spring_vigor["spring_ndvi_p75"] - 0.32) / 0.05), 4)

pixel_hist = (
    df_clean.groupby("pixel_id", as_index=False)
    .agg(
        hist_prob_bare_soil=("prob_bare_obs", "mean"),
        hist_bare_frac=("is_bare_real", "mean"),
        hist_residue_frac=("is_residue_cover", "mean"),
        hist_ndvi_mean=("ndvi", "mean"),
        hist_ndti_mean=("ndti", "mean"),
    )
    .merge(pixel_validity, on="pixel_id", how="left")
    .merge(spring_vigor[["pixel_id", "spring_cover_prob"]], on="pixel_id", how="left")
)

df_grid_merged = df_grid.merge(pixel_hist, on="pixel_id", how="left")
for c in ["hist_prob_bare_soil", "hist_bare_frac", "hist_residue_frac", "hist_ndvi_mean", "hist_ndti_mean", "frac_hls_valida", "spring_cover_prob"]:
    df_grid_merged[c] = df_grid_merged[c].fillna(df_grid_merged[c].median())

slope_cols = [c for c in df_grid_merged.columns if "slope" in c.lower()]
if slope_cols:
    s_col = slope_cols[0]
    s_max = max(float(df_grid_merged[s_col].max()), 1e-6)
    slope_norm = (df_grid_merged[s_col].fillna(0.0) / s_max).clip(0.0, 1.0).to_numpy()
    spatial_mod = 0.90 + 0.20 * slope_norm
else:
    spatial_mod = np.ones(len(df_grid_merged))

# 6. Serie Temporal Semanal Regularizada (Savitzky-Golay + STL)
parcel_daily = (
    df_clean.groupby("date", as_index=False)
    .agg(
        ndvi_mean=("ndvi", "mean"),
        bsi_mean=("bsi", "mean"),
        ndti_mean=("ndti", "mean"),
        prob_bare_soil=("prob_bare_obs", "mean"),
        bare_soil_pct=("is_bare_real", "mean"),
        residue_cover_pct=("is_residue_cover", "mean"),
    )
    .sort_values("date")
)

weekly = (
    parcel_daily.set_index("date")[["ndvi_mean", "bsi_mean", "ndti_mean", "prob_bare_soil", "bare_soil_pct", "residue_cover_pct"]]
    .resample("W-MON")
    .median()
)
weekly = weekly[(weekly.index.year >= 2016) & (weekly.index.year <= 2025)]

is_missing = weekly["ndvi_mean"].isna()
gap_len = is_missing.groupby((~is_missing).cumsum()).transform("sum")
weekly["imputed_flag"] = np.where(~is_missing, 0, np.where(gap_len <= 3, 1, 2))

for col in ["ndvi_mean", "bsi_mean", "ndti_mean", "prob_bare_soil", "bare_soil_pct", "residue_cover_pct"]:
    weekly[col] = weekly[col].interpolate(method="pchip", limit=3, limit_direction="both")

woy = weekly.index.isocalendar().week.astype(int)
for col in ["ndvi_mean", "bsi_mean", "ndti_mean", "prob_bare_soil", "bare_soil_pct", "residue_cover_pct"]:
    if weekly[col].isna().any():
        weekly[col] = weekly[col].fillna(weekly.groupby(woy)[col].transform("median")).bfill().ffill()

weekly = weekly.reset_index()
n = len(weekly)
win = min(7, n if n % 2 == 1 else n - 1)
weekly["ndvi_smooth"] = savgol_filter(weekly["ndvi_mean"].to_numpy(), window_length=win, polyorder=2)
weekly["bsi_smooth"] = savgol_filter(weekly["bsi_mean"].to_numpy(), window_length=win, polyorder=2)
weekly["ndti_smooth"] = savgol_filter(weekly["ndti_mean"].to_numpy(), window_length=win, polyorder=2)
weekly["prob_bare_smooth"] = np.clip(savgol_filter(weekly["prob_bare_soil"].to_numpy(), window_length=win, polyorder=2), 0.0, 1.0)
weekly["bare_frac_smooth"] = np.clip(savgol_filter(weekly["bare_soil_pct"].to_numpy(), window_length=win, polyorder=2), 0.0, 1.0)
weekly["residue_frac_smooth"] = np.clip(savgol_filter(weekly["residue_cover_pct"].to_numpy(), window_length=win, polyorder=2), 0.0, 1.0)

stl = STL(weekly["ndvi_smooth"].to_numpy(), period=52, seasonal=13, trend=79, robust=True).fit()
weekly["ndvi_trend"], weekly["ndvi_seasonal"], weekly["ndvi_resid"] = stl.trend, stl.seasonal, stl.resid

d_ndvi = np.gradient(weekly["ndvi_smooth"].to_numpy())
weekly["pheno_stage"] = np.select(
    [
        (weekly["prob_bare_smooth"] >= 0.50),
        (weekly["ndti_smooth"] > 0.12) & (weekly["ndvi_smooth"] <= 0.35),
        (d_ndvi > 0.025) & (weekly["ndvi_smooth"] > 0.35),
        (weekly["ndvi_smooth"] >= 0.60),
        (d_ndvi < -0.03) & (weekly["ndvi_smooth"] < 0.60),
    ],
    ["bare_fallow", "residue_mulch", "emergence", "peak_canopy", "senescence"],
    default="vegetative",
)

# 7. Ventanas Críticas de Suelo Desnudo Real
weekly["is_bare_soil"] = (weekly["prob_bare_smooth"] >= 0.50) | (weekly["bare_frac_smooth"] >= 0.50)
labeled_array, num_features = label(weekly["is_bare_soil"].to_numpy())

windows = []
for wid in range(1, num_features + 1):
    sub = weekly[labeled_array == wid]
    s_dt, e_dt = sub["date"].iloc[0], sub["date"].iloc[-1] + pd.Timedelta(days=6)
    days = int((e_dt - s_dt).days + 1)
    if days >= 14:
        windows.append({
            "window_id": wid,
            "year": int(s_dt.year),
            "start_date": s_dt.strftime("%Y-%m-%d"),
            "end_date": e_dt.strftime("%Y-%m-%d"),
            "duration_days": days,
            "mean_prob_bare": round(float(sub["prob_bare_smooth"].mean()), 4),
            "mean_bsi": round(float(sub["bsi_smooth"].mean()), 4),
            "mean_ndti": round(float(sub["ndti_smooth"].mean()), 4),
            "mean_bare_soil_frac": round(float(sub["bare_frac_smooth"].mean()), 4),
        })
windows_df = pd.DataFrame(windows)

# 8. Matriz de Confusión y Reporte de Validación
ref_bare = weekly["date"].dt.month.isin([4, 5, 10, 11]) & (weekly["ndvi_smooth"] < 0.32) & (weekly["ndti_smooth"] <= 0.13)
pred_bare = weekly["is_bare_soil"]
tp = int((pred_bare & ref_bare).sum())
tn = int((~pred_bare & ~ref_bare).sum())
fp = int((pred_bare & ~ref_bare).sum())
fn = int((~pred_bare & ref_bare).sum())
acc = (tp + tn) / max(tp + tn + fp + fn, 1)
prec = tp / max(tp + fp, 1)
rec = tp / max(tp + fn, 1)
f1 = 2 * prec * rec / max(prec + rec, 1e-6)

validation_report = {
    "method": "Spectral Classifier (NDVI + BSI + NDTI + NDSI Snow Mask) vs Agronomic Reference",
    "grid_source_used": str(grid_path.name),
    "grid_pixels_count": int(len(df_grid_merged)),
    "calendar_years_aligned": "2016-2025 (10 years)",
    "frac_hls_valida_parcel": round(frac_hls_valida_parcel, 4),
    "snow_contaminated_obs_masked": int((ndsi > 0.20).sum()),
    "confusion_matrix": {"TP": tp, "TN": tn, "FP": fp, "FN": fn},
    "metrics": {"accuracy": round(acc, 4), "precision": round(prec, 4), "recall": round(rec, 4), "f1_score": round(f1, 4)},
}

# 9. Métricas Probabilísticas por (pixel_id, rotation_id)
if isinstance(rotations_data, dict) and "rotations" in rotations_data:
    rot_list = rotations_data["rotations"]
elif isinstance(rotations_data, list):
    rot_list = rotations_data
elif isinstance(rotations_data, dict) and rotations_data:
    rot_list = [{"rotation_id": k, **(v if isinstance(v, dict) else {"sequence": v})} for k, v in rotations_data.items()]
else:
    rot_list = [
        {"rotation_id": "R1_baseline", "sequence": ["corn", "soybean"]},
        {"rotation_id": "R2_conservation", "sequence": ["corn", "cereal_rye", "soybean", "winter_wheat"]},
        {"rotation_id": "R3_resilient_profit", "sequence": ["corn", "soybean", "alfalfa"]},
    ]

crops_meta = catalog.get("crops", {})
c_ref = 0.31

metric_rows = []
for rot in rot_list:
    rot_id = rot.get("rotation_id") or rot.get("id") or rot.get("name")
    seq = rot.get("sequence") or rot.get("crops") or rot.get("crop_sequence") or ["corn", "soybean"]
    c_vals = [float(crops_meta.get(c, {}).get("rusle_c_factor", c_ref)) for c in seq]
    bonus = sum(0.12 for c in seq if crops_meta.get(c, {}).get("functional_group") in ("cover_crop", "deep_rooted"))
    prot = np.clip((np.mean(c_vals) / c_ref) * (1.0 - min(bonus, 0.45)), 0.15, 1.50)

    prob_bare_px = np.clip(df_grid_merged["hist_prob_bare_soil"].to_numpy() * prot * spatial_mod, 0.01, 0.95)
    bare_frac_px = np.clip(df_grid_merged["hist_bare_frac"].to_numpy() * prot * spatial_mod, 0.01, 0.95)
    ndvi_px = df_grid_merged["hist_ndvi_mean"].to_numpy()

    metric_rows.append(pd.DataFrame({
        "pixel_id": df_grid_merged["pixel_id"].to_numpy(),
        "rotation_id": rot_id,
        "prob_bare_soil": np.round(prob_bare_px, 4),
        "frac_hls_valida": np.round(df_grid_merged["frac_hls_valida"].to_numpy(), 4),
        "spring_cover_prob": np.round(df_grid_merged["spring_cover_prob"].to_numpy(), 4),
        "residue_cover_fraction": np.round(df_grid_merged["hist_residue_frac"].to_numpy(), 4),
        "bare_soil_fraction_rot": np.round(bare_frac_px, 4),
        "bare_soil_days_yr": np.round(prob_bare_px * 365.0, 1),
        "canopy_cover_efficiency": np.round(1.0 - prob_bare_px, 4),
        "pheno_vulnerability_idx": np.round(np.clip(0.70 * prob_bare_px + 0.30 * (1.0 - ndvi_px), 0.0, 1.0), 4),
    }))

metrics_pheno_df = pd.concat(metric_rows, ignore_index=True)

# 10. Guardar entregables definitivos en ESTADISTICO 1 y data/
for out_folder in [EST1_DIR, DATA_DIR]:
    if out_folder.exists():
        metrics_pheno_df.to_parquet(out_folder / "metrics_phenology.parquet", index=False)
        weekly.to_parquet(out_folder / "phenology_weekly.parquet", index=False)
        windows_df.to_parquet(out_folder / "bare_soil_windows.parquet", index=False)
        (out_folder / "phenology_validation.json").write_text(
            json.dumps(validation_report, indent=2), encoding="utf-8"
        )
        print(f"Entregables definitivos guardados en: {out_folder}")

print("\n=== RESUMEN DEFINITIVO ESTADÍSTICO 1 (v3) ===")
print(f"Grilla usada: {grid_path.name} ({len(df_grid_merged)} píxeles) | Filas metrics_phenology: {len(metrics_pheno_df)}")
print(f"Fracción HLS válida: {frac_hls_valida_parcel:.4f} | Ventanas críticas: {len(windows_df)} | F1: {validation_report['metrics']['f1_score']}")
print(metrics_pheno_df.head(5))

Carpeta raíz: /content/drive/.shortcut-targets-by-id/17w-HRhuYsYoTirSBYzDsYcqdWB7rM06c/agropulse_xai
Carpeta revisión Matemático: /content/drive/.shortcut-targets-by-id/17w-HRhuYsYoTirSBYzDsYcqdWB7rM06c/agropulse_xai/RUTA_FINAL/1_TRABAJO_EQUIPO/RESULTADOS A REVISAR (MATEMÁTICO)
Carpeta entrega Estadístico 1: /content/drive/.shortcut-targets-by-id/17w-HRhuYsYoTirSBYzDsYcqdWB7rM06c/agropulse_xai/RUTA_FINAL/1_TRABAJO_EQUIPO/RESULTADOS A REVISAR (ESTADISTICOS)/ESTADISTICO 1

=== GRILLA ARMONIZADA SELECCIONADA ===
Archivo: /content/drive/.shortcut-targets-by-id/17w-HRhuYsYoTirSBYzDsYcqdWB7rM06c/agropulse_xai/data/parquet_VERIFICADO/harmonized_grid_30m.parquet
Píxeles: 1200 | Columnas: ['pixel_id', 'lon', 'lat', 'elevation_m', 'slope_pct', 'ssurgo_awc', 'smap_sm_30m', 'gpm_precip_30m', 'power_tmax_30m']

Entregables definitivos guardados en: /content/drive/.shortcut-targets-by-id/17w-HRhuYsYoTirSBYzDsYcqdWB7rM06c/agropulse_xai/RUTA_FINAL/1_TRABAJO_EQUIPO/RESULTADOS A REVISAR (ESTADISTICOS)/E